## RUN FIRST TIME ONLY

The solar-dynamo model is a *stochastic delay differential equation* (SDDE), d the
reference implementation provided (`SolarDynamo.jl`) is written in Julia,
which has mature SDDE solvers (`DelayDiffEq` + `StochasticDiffEq`). Instead of re-implementing
the physics in Python (error-prone, and hard to validate), we keep the Julia simulator as-is
and *wrap* it from Python via `juliacall`. All the machine-learning part (the autoencoder)
is instead written in PyTorch, the framework covered in the course.

In [ ]:
!pip install juliacall

In [ ]:
%%capture
import juliacall
jl = juliacall.newmodule('S')
jl.seval('import Pkg; Pkg.add(["DelayDiffEq", "StochasticDiffEq", "SpecialFunctions", "StaticArrays", "FFTW"])')

## RUN ONLY IF PYTORCH IS INSTALLED WITHOUT GPU DEPENDENCIES

In [ ]:
import os

# disable search for CPU triton
os.environ["TORCHINDUCTOR_REDUCING_RECOMPILE_PROFILING"] = "0"
os.environ["TRITON_CPU_BACKEND"] = "0"

---

# Solar Dynamo - ENCA

**Goal of the project.** Learn *minimal, near-sufficient summary statistics* for the
5-parameter stochastic solar-dynamo model, to be used (in principle) for simulation-based
Bayesian inference (ABC). In the original inference paper the summary statistics were
hand-picked Fourier components (~20 of them) — an arbitrary choice. Here we let a neural
network *learn* them.

**The method (ENCA = Explicit Noise Conditional Autoencoder, Albert et al. 2022).** It is
an autoencoder in which:

- the **encoder** compresses a simulated time series $x$ into a small vector of summaries $s$;
- the **decoder** receives $s$ **plus the exact noise realisation** $\epsilon$ that was used
  inside the simulator, and must reconstruct $x$.

Because the decoder is *given* the noise for free, the encoder has no incentive to waste
its few summary dimensions encoding noise: it only needs to store the information that comes
from the parameters $\theta$. By construction, $s$ then approximates a minimal set of
summary statistics for $\theta$.

**Why it is hard for this model.** The dynamo SDDE has chaotic regimes (chaos induced by the
time delay $T$). The autoencoder loss is a *pointwise* comparison between $x$ and $\hat{x}$:
reproducing a chaotic trajectory point-by-point is essentially impossible, so we expect the
plain time-domain approach to break down as we approach chaos — this motivates the Fourier
experiments in the second half of the notebook.

## Imports and Global Settings

In [ ]:
from juliacall import Main as jl
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from IPython.display import display, Image
import io

jl.include("SolarDynamo.jl")
jl.seval("using .SolarDynamo")
sn = jl.SolarDynamo.sn
sn_from_noise = jl.SolarDynamo.sn_from_noise

# check whether to use cpu or gpu
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


TOBS = 200 # length of the observed time series
TWARMUP = 200 # initial transient that is discarded
DT = 0.1 # internal Euler-Maruyama integration step of the SDDE solver

# tau : diffusive decay time of the magnetic field (sets the time scale, dimensionful);
# T : time delay of the Babcock-Leighton loop (source of chaos!);
# Nd : dynamo number, ratio of driving vs decay (the key bifurcation parameter);
# sigma : amplitude of the stochastic forcing (turbulent plasma fluctuations);
# Bmax  : upper threshold of the alpha-effect window; it only rescales the y-axis.
PARAM_NAMES = ["tau", "T", "Nd", "sigma", "Bmax"]

# USE_FOURIER / USE_FOURIER_NOISE / USE_FNO: switches for the experiments of sections 6-8. They change the representation of the data (time domain vs log-magnitude spectrum).
USE_FOURIER = False
USE_FOURIER_NOISE = False
FOURIER_SIZE = TOBS // 2 + 1
USE_FNO = False

## Simulator

**Key design decision: the noise is generated *outside* the simulator.**
ENCA needs the *bare* noise realisation $\epsilon$ (i.e. independent of $\theta$) to feed
the decoder. The original Julia function `sn` generates the Wiener increments internally and
does not return them.

Our first attempt was to "reverse-engineer" the noise by running the same seed with
$\sigma=0$ and taking the difference between stochastic and deterministic run. But, this is wrong: the noise enters *at every integration step* and propagates
through the (delayed) dynamics, so the residual is not the bare noise — worse, it leaks
information about $x$ itself, making the decoder's task artificially easy and destroying the
very mechanism that forces the encoder to extract only parameter information.

The correct solution is `sn_from_noise`: we draw
the standard-normal increments `eps_dt` in Python, pass them to Julia, and Julia integrates
the SDDE with exactly that noise (Euler-Maruyama with ring buffer for the delayed state).
The returned observable is $x = B^2$, the proxy for the sunspot number.

**Noise downsampling for the decoder.** The solver uses $DT=0.1$, so one simulation consumes
4000 noise increments for 200 observed points. Feeding all 4000 raw increments to the decoder
was tried and performed very poorly; then we aggregate
the 10 increments within each unit time interval by summing them and dividing by $\sqrt{10}$
(sum of $k$ iid N(0,1) has std $\sqrt{k}$, so `eps_obs` stays standard-normal). This gives a noise
series aligned 1:1 with the observed grid. It discards some information, but empirically it
works much better — and it is statistically natural: it is exactly the Brownian increment
over one observation interval, rescaled to unit variance.

In [ ]:
def simulate(theta, seed=None):
    tau, T, Nd, sigma, Bmax = theta
    T = round(T / DT) * DT

    if seed is not None:
        np.random.seed(seed)

    # generate noise
    Ndt = int( (TWARMUP+TOBS) / DT ) # = 4000 if TWARMUP=200,TOBS=200,DT=0.1
    eps_dt = np.random.randn(Ndt).astype(np.float64)

    # convert to julia types
    theta_jl = jl.Vector[jl.Float64]([tau, T, Nd, sigma, Bmax])
    eps_jl = jl.Vector[jl.Float64](eps_dt)

    # stochastic simulation
    x = sn_from_noise(theta_jl, eps_jl, Twarmup=TWARMUP, Tobs=TOBS, dt=DT)
    x = np.array(x, dtype=np.float32)

    # extract from eps_dt just the non-warmup portion
    warmup_steps = int(TWARMUP / DT)
    # compute the sum of noise per block
    k = int(1.0/DT)
    eps_obs = np.array([
        np.sum( eps_dt[warmup_steps+i*k : warmup_steps + (i+1)*k] )
        for i in range(TOBS)], dtype=np.float32)
    eps_obs /= np.sqrt(k)

    return x, eps_obs

# Fourier representation (used from section 6 onward):
# we keep log1p(|rfft(x)|) as data and store the phase separately.
# - log1p compresses the huge dynamic range of the spectrum (the dominant peak is orders of
#   magnitude above the background), otherwise the MSE loss would only "see" the main peak.
# - The phase is NOT given to the network: the parameter information sits in the magnitude
#   spectrum (location/height of the dominant peak + shape of the background), while the phase
#   mostly reflects the particular realisation. We keep it only to back-project reconstructions
#   to the time domain for visual comparison (fourier_to_time).
def to_fourier(X):
    # compute Fourier transform
    F_complex = np.fft.rfft(X, axis=1)
    # keep both amplitude of frequencies and phase
    F_mag = np.abs(F_complex)
    Phase = np.angle(F_complex)
    # compress the dynamic range
    F = np.log1p(F_mag)
    
    return F.astype(np.float32), Phase.astype(np.float32)

def fourier_to_time(X_f_norm, stats, phase=None):
    # undo normalization
    X_f = X_f_norm * stats["x_std"] + stats["x_mean"]
    # undo log1p
    X_f = np.expm1(X_f)
    
    if phase is not None:
        # recombine magnitude and original phase
        F_complex = X_f * np.exp(1j * phase)
        X_time = np.fft.irfft(F_complex, n=TOBS, axis=-1)
    else:
        # zero-phase IFFT
        X_time = np.fft.irfft(X_f, n=TOBS, axis=-1)
    
    return X_time.astype(np.float32)

def get_output_size():
    return FOURIER_SIZE if USE_FOURIER else TOBS

## Dataset Generation and I/O

Each experiment is defined by a `fixed` dict (which parameters are frozen, which vary) and a
`priors` dict (uniform prior ranges for the varying ones). For every sample we draw
theta ~ $Uniform(prior)$, simulate, and store (x, eps, theta_varying).

Practical points that should be mentioned:
- **Normalization**: x and theta are standardized with statistics computed on the *training*
  set only, and the same stats are reused for the test set (standard practice — the network
  must not see test-set information, and inputs at test time must live on the same scale it
  was trained on). The noise is already ~$N(0,1)$ by construction, so it is left untouched.
- **Caching**: datasets are saved to disk with a filename that encodes n_samples, fixed
  values, priors and representation (time/fourier), so each configuration is simulated only
  once. Same for trained models. This is why the experiment cells below run with
  `train_model=False`: they reload models trained previously (mostly on Google Colab GPUs,
  since the bigger runs use up to 150k simulations).

In [ ]:
def make_basename(n_samples, fixed, prior):
    fixed_text = "-".join([str(v) for v in fixed.values()])
    prior_text = "_".join([
        f"{k}={prior[k][0]},{prior[k][1]}"
        for k in PARAM_NAMES
        if fixed.get(k) is None and isinstance(prior.get(k), tuple)
    ])
    fourier_text = "fourier" if USE_FOURIER else "time"
    noise_text   = "__fnoise" if (USE_FOURIER and USE_FOURIER_NOISE) else ""
    return f"{n_samples}__{fixed_text}__{prior_text}__{fourier_text}{noise_text}"


def generate_dataset(n_samples, fixed, priors, stats=None):
    varying_names = [k for k,v in fixed.items() if v is None]
    n_params = len(varying_names)

    # create the dataset
    X = np.zeros((n_samples, TOBS), dtype=np.float32)
    Noise = np.zeros((n_samples, TOBS), dtype=np.float32)
    Params = np.zeros((n_samples, n_params), dtype=np.float32)

    for i in range(n_samples):
        theta = {}
        for name in PARAM_NAMES:
            if fixed[name] is not None:
                theta[name] = fixed[name]
            else:
                low, high = priors[name]
                theta[name] = np.random.uniform(low,high)

        x, noise = simulate([theta[k] for k in PARAM_NAMES])
        X[i] = x
        Noise[i] = noise
        Params[i] = [theta[k] for k in varying_names]

        if (i+1)%50 == 0:
            print(f"Generated {i+1}/{n_samples} samples", end="\r")

    print()

    Phase = None
    # check if X in Fourier Space or Time Space
    if USE_FOURIER:
       X, Phase = to_fourier(X)
       if USE_FOURIER_NOISE:
           # represent each noise realisation as log1p(|rfft(noise)|)
           Noise = np.log1p(
               np.abs(np.fft.rfft(Noise, axis=1))
           ).astype(np.float32)


    if stats is None:
      # normalize x
      x_mean = X.mean()
      x_std = X.std()

      # don't normalize noise because already normalized
      n_mean = 0
      n_std = 1

      # normalize parameters
      p_mean = Params.mean(axis=0)
      p_std  = Params.std(axis=0)
      p_std[p_std == 0] = 1.0

      stats = {
          "x_mean": x_mean, "x_std": x_std,
          "n_mean": n_mean, "n_std": n_std,
          "p_mean": p_mean, "p_std": p_std
      }

    X = (X - stats["x_mean"]) / stats["x_std"]
    Params = (Params - stats["p_mean"]) / stats["p_std"]

    return X, Noise, Params, stats, Phase

def save_dataset(X, Noise, Params, stats, Phase, n_samples, fixed, priors, folder="data"):
    os.makedirs(folder, exist_ok=True)
    base = make_basename(n_samples, fixed, priors)

    np.save(os.path.join(folder, f"X_{base}.npy"), X)
    np.save(os.path.join(folder, f"Noise_{base}.npy"), Noise)
    np.save(os.path.join(folder, f"Params_{base}.npy"), Params)
    np.save(os.path.join(folder, f"stats_{base}.npy"), stats, allow_pickle=True)
    if Phase is not None:
        np.save(os.path.join(folder, f"Phase_{base}.npy"), Phase)
    print(f"Saved → {folder}/{base}\n")

def load_dataset(n_samples, fixed, priors, folder="data"):
    base = make_basename(n_samples, fixed, priors)
    try:
        X = np.load(os.path.join(folder, f"X_{base}.npy"))
        Noise = np.load(os.path.join(folder, f"Noise_{base}.npy"))
        Params = np.load(os.path.join(folder, f"Params_{base}.npy"))
        stats = np.load(os.path.join(folder, f"stats_{base}.npy"), allow_pickle=True).item()
        try:
            Phase = np.load(os.path.join(folder, f"Phase_{base}.npy"))
        except FileNotFoundError:
            Phase = None
        print(f"Loaded ← {folder}/{base}")
        return X, Noise, Params, stats, Phase
    except FileNotFoundError:
        print(f"Not found: {base}. Generate it first.")
        return None

def generate_train_test_datasets(fixed, priors,
                                 n_train=5000, n_test=1000,
                                 folder="data"):
    # check if exists
    base = make_basename(n_train, fixed, priors)
    if os.path.isfile(os.path.join(folder, f"X_{base}.npy")):
        print("Dataset already exists, skipping generation.")
        return

    # generate and save a training dataset
    print("Generating training data...")
    X_tr, N_tr, P_tr, stats, Phase_tr = generate_dataset(n_train, fixed, priors)
    save_dataset(X_tr, N_tr, P_tr, stats, Phase_tr, n_train, fixed, priors, folder)

    # generate and save a test dataset
    print("Generating test data...")
    X_te, N_te, P_te, _, Phase_te = generate_dataset(n_test, fixed, priors, stats=stats)
    save_dataset(X_te, N_te, P_te, stats, Phase_te, n_test, fixed, priors, folder)
    print("Done.")

def load_train_test_datasets(fixed, priors,
                             n_train=5000, n_test=1000,
                             folder="data"):
    # load datasets
    result_tr = load_dataset(n_train, fixed, priors, folder)
    result_te = load_dataset(n_test,  fixed, priors, folder)
    if result_tr is None or result_te is None:
        return None

    X_tr, N_tr, P_tr, stats, Phase_tr = result_tr
    X_te, N_te, P_te, _, Phase_te     = result_te
    return {
        "X_train": X_tr, "N_train": N_tr, "P_train": P_tr, "stats": stats, "Phase_train": Phase_tr,
        "X_test":  X_te, "N_test":  N_te, "P_test":  P_te, "Phase_test": Phase_te
    }

## ENCA Architecture

**Encoder (1D CNN).** Two convolutional blocks followed by a projection to `n_summary`
channels and a *global average pooling* over time. The CNN extracts local oscillation
patterns; the global pooling makes the summaries (approximately) translation-invariant,
which is what we want: the parameters determine the *statistics* of the cycle, not where
a given cycle happens to sit in the window.

**Decoders — three variants, reflecting the evolution of the project:**

1. `Decoder` (bidirectional LSTM, time domain): the decoder receives at each time step the
   summaries s (tiled) and the noise increment of that step, and emits the reconstruction.
   An LSTM is the natural choice here, because the model is a *delay* equation: the value at
   time t depends on the past (the state at t-T), so a recurrent network with memory mimics
   the structure of the simulator. It is *bidirectional* following the reference paper:
   knowing the future noise also helps to reconstruct the present. A plain fully-connected decoder was tried first and
   discarded: with the correct bare noise it simply does not capture the waveform.
2. `DecoderCNN` (Fourier domain): once the data live in frequency space there is no temporal
   recurrence to exploit, an LSTM no longer makes sense. A CNN over the frequency
   axis maps [projected summaries, noise] → log-magnitude spectrum.
3. `DecoderFNO` (Fourier Neural Operator): suggested by the professor via email after
   meeting 6. Each FNO block performs a *global* mixing in frequency space (a learned complex
   matrix per retained mode) plus a local pointwise bypass. FNOs are designed precisely to
   learn solution operators of differential equations, i.e. the mapping
   (noise realisation, parameters) → trajectory, which is literally our decoder's job.

The full ENCA simply chains encoder and the selected decoder; the loss (next section) ties
the first `n_params` summary dimensions to the true parameters.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, n_summary):
        super().__init__()

        self.net = nn.Sequential(
            # block 1
            nn.Conv1d(1,  16, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv1d(16, 16, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool1d(2), # halve the time dimension

            # block 2
            nn.Conv1d(16, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv1d(32, 32, kernel_size=3, padding=1), nn.ReLU(),

            # project to n_summary channels
            nn.Conv1d(32, n_summary, kernel_size=3, padding=1),
        )

        # average over time: (batch,n_summary,TOBS)->(batch,n_summary,1)
        self.pool = nn.AdaptiveAvgPool1d(1)

    def forward(self, x):
        # x: (batch,TOBS) -> add channel dim -> (batch,1,TOBS)
        x = x.unsqueeze(1)
        # x -> (batch,n_summary,TOBS/2)
        x = self.net(x)
        # x -> (batch,n_summary,1)
        x = self.pool(x)
        # x -> (batch,n_summary)
        x = x.squeeze(-1)

        return x


class Decoder(nn.Module):
    def __init__(self, n_summary):
        super().__init__()

        self.lstm = nn.LSTM(input_size=n_summary + 1, hidden_size=16,
                            num_layers=2, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(32, 1)

    def forward(self, s, noise):
        s_tiled = s.unsqueeze(1).expand(-1, noise.shape[1], -1)
        noise3d = noise.unsqueeze(2)
        inp = torch.cat([s_tiled, noise3d], dim=-1)

        out, _ = self.lstm(inp)
        x_hat = self.fc(out).squeeze(-1)

        # resize output for Fourier or Time Space
        target_len = get_output_size()
        if x_hat.shape[1] != target_len:
            x_hat = nn.functional.interpolate(
                x_hat.unsqueeze(1), size=target_len, mode="linear", align_corners=False
            ).squeeze(1)

        return x_hat


class DecoderCNN(nn.Module):
    def __init__(self, n_summary):
        super().__init__()
        output_size = get_output_size()
        # project summary stats onto the output length
        self.s_proj = nn.Linear(n_summary, output_size)
        # CNN that takes [s_projected, noise] as 2 channels
        self.net = nn.Sequential(
            nn.Conv1d(2, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv1d(32, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv1d(32, 16, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv1d(16,  1, kernel_size=3, padding=1),
        )

    def forward(self, s, noise):
        # s: (batch, n_summary) -> (batch, 1, output_size)
        s_proj = self.s_proj(s).unsqueeze(1)
        # noise: (batch, TOBS) -> (batch, 1, TOBS)
        # if noise length differs from output_size, interpolate
        target_len = get_output_size()
        if noise.shape[1] != target_len:
            noise = nn.functional.interpolate(
                noise.unsqueeze(1), size=target_len, mode="linear", align_corners=False
            ).squeeze(1)
        noise = noise.unsqueeze(1)
        # concat along channel dim -> (batch, 2, output_size)
        x = torch.cat([s_proj, noise], dim=1)
        # (batch, 1, output_size) -> (batch, output_size)
        x = self.net(x).squeeze(1)
        return x
    

class FNOBlock(nn.Module):
    """Single 1-D FNO layer: spectral path + local bypass + GELU activation."""
    def __init__(self, width, n_modes):
        super().__init__()
        self.n_modes = n_modes
        # One learned (width x width) complex matrix per retained mode.
        # Stored as real and imaginary parts separately (PyTorch parameter constraint).
        self.R_real = nn.Parameter(torch.randn(n_modes, width, width) * 0.02)
        self.R_imag = nn.Parameter(torch.randn(n_modes, width, width) * 0.02)
        # Local pointwise bypass (1x1 conv = position-wise linear transform)
        self.W   = nn.Conv1d(width, width, kernel_size=1)
        self.act = nn.GELU()

    def _spectral_mix(self, V):
        """
        V: complex tensor of shape (batch, width, n_freq).
        For each retained mode m: V_out[:, :, m] = R[m] @ V[:, :, m]
        where R[m] is a (width x width) complex matrix.
        """
        k = min(self.n_modes, V.shape[-1])
        R = torch.complex(self.R_real[:k], self.R_imag[:k])  # (k, width, width)
        # V[:, :, :k] : (batch, width_in, k_modes)
        # R            : (k_modes, width_out, width_in)
        # output       : (batch, width_out, k_modes)
        V_out = torch.zeros_like(V)
        V_out[:, :, :k] = torch.einsum('bim,moi->bom', V[:, :, :k], R)
        return V_out

    def forward(self, v):
        # v: (batch, width, T)
        V          = torch.fft.rfft(v, dim=-1)          # → (batch, width, T//2+1)
        V_filtered = self._spectral_mix(V)              # mix low-freq modes
        v_spec     = torch.fft.irfft(V_filtered,
                                     n=v.shape[-1], dim=-1)   # → (batch, width, T)
        return self.act(v_spec + self.W(v))             # combine + activate


class DecoderFNO(nn.Module):
    """
    FNO-based decoder.
    Maps (s: summary stats, noise: noise time series) → x_hat.

    Architecture:
      1. Lift: project [noise | s_tiled] from (1+n_summary) channels → width channels
      2. N FNO blocks: global spectral mixing + local bypass at each layer
      3. Project: width channels → 1 channel (the reconstruction)

    Works in both time space (USE_FOURIER=False) and Fourier space (USE_FOURIER=True)
    because it uses get_output_size() internally.
    """
    def __init__(self, n_summary, width=32, n_modes=16, n_layers=4):
        super().__init__()
        self.output_size = get_output_size()
        in_channels      = 1 + n_summary     # 1 noise channel + n_summary summary dims

        self.lift   = nn.Conv1d(in_channels, width, kernel_size=1)
        self.blocks = nn.ModuleList([
            FNOBlock(width, n_modes) for _ in range(n_layers)
        ])
        self.project = nn.Sequential(
            nn.Conv1d(width, 16, kernel_size=1), nn.GELU(),
            nn.Conv1d(16,    1, kernel_size=1),
        )

    def forward(self, s, noise):
        # s:     (batch, n_summary)
        # noise: (batch, TOBS)
        T = self.output_size

        # Resize noise to match output length (handles Fourier vs time mismatch)
        if noise.shape[1] != T:
            noise = nn.functional.interpolate(
                noise.unsqueeze(1), size=T, mode="linear", align_corners=False
            ).squeeze(1)

        # Build input field: noise + summary broadcast across time
        # noise.unsqueeze(1)       : (batch, 1, T)
        # s.unsqueeze(-1).expand…  : (batch, n_summary, T)
        v = torch.cat([noise.unsqueeze(1),
                       s.unsqueeze(-1).expand(-1, -1, T)], dim=1)  # (batch, 1+n_summary, T)

        v = self.lift(v)           # (batch, width, T)
        for block in self.blocks:
            v = block(v)           # (batch, width, T)
        return self.project(v).squeeze(1)   # (batch, T)


class ENCA(nn.Module):
    def __init__(self, n_summary):
        super().__init__()

        self.encoder = Encoder(n_summary)
        if USE_FNO:
          self.decoder = DecoderFNO(n_summary)
        elif USE_FOURIER:
          self.decoder = DecoderCNN(n_summary)
        else:
          self.decoder = Decoder(n_summary)

    def forward(self, x, noise):
        s = self.encoder(x)
        x_hat = self.decoder(s, noise)
        return s, x_hat

## Training and Evaluation

**Loss = reconstruction + regression.**
- *Reconstruction*: MSE(x_hat, x) — the autoencoder part. We also experimented with a
  chi-squared-like loss (as in the reference paper) but it was unstable on our normalized
  data (near-zero denominators) and consistently worse than plain MSE, so MSE was kept.
- *Regression*: MSE(s[:, :n_params], theta) — pins the first `n_params` summary dimensions
  to the (normalized) true parameters. This makes the summaries directly interpretable as
  parameter estimators and lets us check the quality with simple s_i vs theta_i scatter plots.
  The remaining dimensions are left free for whatever extra information the reconstruction
  needs (e.g. which dynamical regime the realisation is in — see the cluster analysis below).

**How many summaries?** The Bernstein-von Mises argument: with finitely many
effective replicas in a single time series, parameter estimators alone are generally not
sufficient, so one should allow a few *more* summaries than parameters. We follow the
professor's rule of thumb: `n_summary = n_params + 2` by default (and we test
`n_summary = n_params` in section 6d to show the extra dimensions are actually needed).

Training details: Adam, lr=1e-3 with exponential decay (gamma=0.99) for stable convergence,
200 epochs, batch size 64.

**Evaluation plots** (produced by `evaluate` for every experiment):
1. Reconstructions x_hat vs x on test samples (in Fourier mode: both the spectrum and the
   back-projected time series, recombined with the true phase);
2. Regression check: s_i vs true parameter, should lie on the y=x line;
3. Latent space: scatter of the summary vector, colored by the true parameter — this is where
   the most interesting structure (clusters/manifolds) shows up;
4. Extra (free) summary dimensions vs the parameter-mapped ones.

In [ ]:
def train(X, Noise, Params, n_summary=None, epochs=200,
          batch_size=64, lr=1e-3):

    n_params = Params.shape[1]
    n_summary = n_summary or n_params+2

    # create the iterable dataset
    dataset = TensorDataset(
        torch.tensor(X),
        torch.tensor(Noise),
        torch.tensor(Params)
    )
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    # define model, optimizer and loss function
    model = ENCA(n_summary).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    # change LR at each epoch with a scheduler
    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.99)
    loss_func = nn.MSELoss()

    loss_history = []
    model.train()

    for epoch in range(epochs):
        total_loss = [0, 0, 0]
        n_batches = 0

        for x_batch, n_batch, p_batch in loader:
            x_batch = x_batch.to(DEVICE)
            n_batch = n_batch.to(DEVICE)
            p_batch = p_batch.to(DEVICE)

            # get output from the model
            s, x_hat = model(x_batch, n_batch)

            # calculate reconstruction loss and regression loss
            loss_recon = loss_func(x_hat, x_batch)
            loss_regr = loss_func(s[:, :n_params], p_batch)
            loss_tot = loss_recon + loss_regr

            # backpropagate loss
            optimizer.zero_grad()
            loss_tot.backward()
            optimizer.step()

            total_loss[0] += loss_tot.item()
            total_loss[1] += loss_recon.item()
            total_loss[2] += loss_regr.item()
            n_batches += 1

        # save dataset loss
        avg_loss = [l/n_batches for l in total_loss]
        loss_history.append(avg_loss)

        # update learning rate
        scheduler.step()

        if (epoch+1)%20 == 0:
            lr_now = scheduler.get_last_lr()[0]
            print(f"Epoch {epoch+1:3d}, Loss: {avg_loss[0]:.4f}, LR: {lr_now:.2e}")

    # plot loss curves
    loss_history = torch.tensor(loss_history)

    return model, loss_history

def plot_loss(loss_history):
    loss_history = loss_history.cpu()
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(loss_history[:,0], label="Total", lw=3)
    ax.plot(loss_history[:,1], linestyle="--",
            label="Reconstruction", lw=3)
    ax.plot(loss_history[:,2], linestyle=":",
            label="Regression", lw=3)
    ax.set_yscale("log")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE Loss")
    ax.set_title(f"Training Loss Curves | $MSE_{{{len(loss_history)}}}$ = {loss_history[-1,0].item():.4f}")
    ax.grid(True, alpha=0.3)
    ax.legend(frameon=True, fancybox=True, shadow=True)
    plt.tight_layout()
    plt.show()

def evaluate(model, X, Noise, Params, stats, fixed, priors, Phase=None, show_plots=True):
    varying_names = [k for k,v in fixed.items() if v is None]
    n_params = len(varying_names)

    # prediction
    model.eval()
    with torch.no_grad():
        s, x_hat = model(
            torch.tensor(X).to(DEVICE),
            torch.tensor(Noise).to(DEVICE)
        )

    # convert to numpy
    s = s.cpu().numpy()
    x_hat = x_hat.cpu().numpy()

    # un-noralize paramters and summaries to real values
    Params_real = Params*stats["p_std"] + stats["p_mean"]
    s_real = s.copy()
    s_real[:, :n_params] = s[:, :n_params] * stats["p_std"] + stats["p_mean"]
    
    if not show_plots:
        return s, x_hat

    # 1) plot reconstruction (3 random examples)
    idx = np.random.choice(len(X), size=3, replace=False)
    if USE_FOURIER:
        # plot both Fourier space and back-projected time space
        fig, axes = plt.subplots(3, 2, figsize=(16, 8))
        for row, i in zip(axes, idx):
            title = ", ".join([f"{p}={Params_real[i,j]:.2f}"
                               for j,p in enumerate(varying_names)])
            X_disp     = X[i]     * stats["x_std"] + stats["x_mean"]
            x_hat_disp = x_hat[i] * stats["x_std"] + stats["x_mean"]
            mse_f = np.mean((X_disp - x_hat_disp)**2)

            # Fourier space
            row[0].plot(X_disp,     label="Original",      lw=2)
            row[0].plot(x_hat_disp, label="Reconstructed", lw=2, ls="--")
            row[0].set_title(f"{title} | MSE(Fourier) = {mse_f:.4f}")
            row[0].set_ylabel("log|FFT(x)|")
            row[0].grid(True, alpha=0.3)

            # time space with exact phase
            phase_orig = Phase[i:i+1] if Phase is not None else None
            X_t     = fourier_to_time(X[i:i+1],     stats, phase=phase_orig)[0]
            X_hat_t = fourier_to_time(x_hat[i:i+1], stats, phase=phase_orig)[0]
            mse_t = np.mean((X_t - X_hat_t)**2)
            
            row[1].plot(X_t,     label="Original",      lw=2)
            row[1].plot(X_hat_t, label="Reconstructed", lw=2, ls="--")
            row[1].set_title(f"Back-projected | MSE(time) = {mse_t:.4f}")
            row[1].set_ylabel("x")
            row[1].grid(True, alpha=0.3)

        axes[0][0].legend(loc="upper right")
        axes[0][1].legend(loc="upper right")
        axes[-1][0].set_xlabel("frequency bin")
        axes[-1][1].set_xlabel("time step")

    else:
        fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
        for ax, i in zip(axes, idx):
            title = ", ".join([f"{p}={Params_real[i,j]:.2f}"
                               for j,p in enumerate(varying_names)])
            X_disp     = X[i]     * stats["x_std"] + stats["x_mean"]
            x_hat_disp = x_hat[i] * stats["x_std"] + stats["x_mean"]
            mse_val = np.mean((X_disp - x_hat_disp)**2)
            ax.plot(X_disp,     label="Original x", lw=3)
            ax.plot(x_hat_disp, label="Reconstructed", lw=3, ls="--")
            ax.set_title(f"{title} | MSE = {mse_val:.4f}")
            ax.set_ylabel("x")
            ax.grid(True, alpha=0.3)
        axes[0].legend(loc="upper right")
        axes[-1].set_xlabel("time step")

    plt.suptitle("Reconstruction of data")
    plt.tight_layout()
    _show_static(fig)

    # 2) plot s_j vs true param
    fig, axes = plt.subplots(1, n_params, figsize=(4*n_params,4))
    if n_params == 1:
        axes = [axes]
    for i,(ax,name) in enumerate(zip(axes,varying_names)):
        # scatter plot
        ax.scatter(s_real[:,i], Params_real[:,i], alpha=0.4,
                   edgecolors="none", s=8)
        # y=x line
        low, high = priors[name]
        ax.plot([low, high], [low, high], c="#222222", ls="--",
                lw=2, label="$y=x$")

        ax.set_ylabel(f"true {name}")
        ax.set_xlabel(f"$s_{i}$")
        ax.set_title(f"$s_{i}$ vs {name}")
        ax.set_xlim(low, high)
        ax.set_ylim(low, high)
        ax.set_aspect("equal")
        ax.grid(True, alpha=0.3)
    axes[0].legend()
    plt.suptitle("Regression check")
    plt.tight_layout()
    _show_static(fig)

    # 3) plot latent space
    if n_params == 1:
        fig, ax = plt.subplots(figsize=(6,5))
        sc = ax.scatter(s[:,0], s[:,1], c=Params_real[:,0],
                        cmap="plasma", alpha=0.5, s=15, edgecolors="none")
        plt.colorbar(sc, ax=ax, label=varying_names[0])
        ax.set_ylabel("$s_1$")
        ax.set_xlabel("$s_0$")
        ax.grid(True, alpha=0.3)
        plt.suptitle(f"Latent space colored by {varying_names[0]}")
        plt.tight_layout()
        _show_static(fig)
    elif n_params==2:
        fig = plt.figure(figsize=(6, 5))
        ax = fig.add_subplot(projection="3d")
        sc = ax.scatter(s[:,0], s[:,1], s[:,2], c=Params_real[:,0],
                        cmap="plasma", alpha=0.3, s=15, edgecolors="none")
        plt.colorbar(sc, ax=ax, label=varying_names[0])
        ax.set_zlabel("$s_2$")
        ax.set_ylabel("$s_1$")
        ax.set_xlabel("$s_0$")
        plt.suptitle(f"Latent space colored by {varying_names[0]}")
        plt.show()
    
        plot_extra_summaries(s, Params_real, varying_names, n_params)
        
    else:
        plot_extra_summaries(s, Params_real, varying_names, n_params)

    return s, x_hat

def plot_extra_summaries(s, Params_real, varying_names, n_params):
    n_extra = s.shape[1] - n_params
    if n_extra <= 0:
        return
 
    fig, axes = plt.subplots(n_extra, n_params,
                              figsize=(4 * n_params, 3.5 * n_extra),
                              squeeze=False)
 
    for ei in range(n_extra):
        si = n_params + ei
        for pi in range(n_params):
            ax = axes[ei, pi]
            sc = ax.scatter(s[:, pi], s[:, si],
                            c=Params_real[:, pi], cmap="plasma",
                            alpha=0.4, s=8, edgecolors="none")
            plt.colorbar(sc, ax=ax, label=varying_names[pi])
            ax.set_xlabel(f"$s_{{{pi}}}$  ({varying_names[pi]})")
            ax.set_ylabel(f"$s_{{{si}}}$  (free {ei + 1})")
            ax.grid(True, alpha=0.3)
 
    plt.suptitle("Extra summary dimensions vs parameter-mapped summaries",
                 fontsize=12)
    plt.tight_layout()
    _show_static(fig)

def _show_static(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', dpi=120)
    buf.seek(0)
    plt.close(fig)
    display(Image(buf.read()))

## Model I/O

Saving/loading of trained models, with consistency checks: a checkpoint stores the
representation flags (USE_FOURIER, USE_FNO) it was trained with, and loading fails if
the current global flags don't match — this prevents silently evaluating a Fourier-trained
model on time-domain data.

In [ ]:
def save_model(model, loss_history, fixed, priors, n_summary, folder="models"):
    os.makedirs(folder, exist_ok=True)
    fno_text = "__fno" if USE_FNO else ""
    base = make_basename(0, fixed, priors).split("__", 1)[1]
    path = os.path.join(folder, f"enca__{base}__{n_summary}summary{fno_text}.pt")
    torch.save({
        "model_state": model.state_dict(),
        "loss_history": loss_history,
        "use_fourier": USE_FOURIER,
        "use_fno": USE_FNO
    }, path)
    print(f"Model saved → {path}")
    return path

def load_model(fixed, priors, n_summary, folder="models"):
    base = make_basename(0, fixed, priors).split("__", 1)[1]
    fno_text = "__fno" if USE_FNO else ""
    path = os.path.join(folder, f"enca__{base}__{n_summary}summary{fno_text}.pt")

    checkpoint = torch.load(path, map_location=DEVICE)
    saved_fourier = checkpoint.get("use_fourier", None)
    if saved_fourier is not None and saved_fourier != USE_FOURIER:
        raise ValueError(
            f"Model was saved with USE_FOURIER={saved_fourier}, "
            f"but current USE_FOURIER={USE_FOURIER}. Set USE_FOURIER={saved_fourier} before loading."
        )
    saved_fno = checkpoint.get("use_fno", None)
    if saved_fno is not None and saved_fno != USE_FNO:
        raise ValueError(
            f"Model was saved with USE_FNO={saved_fno}, "
            f"but current USE_FNO={USE_FNO}. Set USE_FNO={saved_fno} before loading."
        )
    model = ENCA(n_summary).to(DEVICE)
    model.load_state_dict(checkpoint["model_state"])

    model.eval()
    print(f"Model loaded ← {path}")
    return model, checkpoint["loss_history"]

## Experiment Runner

`experiment(...)` is the single entry point used by all sections below:
generate (or load) the datasets → train (or load) the model → evaluate on the test set.
Each experiment is thus fully specified by: which parameters vary, the prior ranges,
the data representation (time/Fourier), the decoder type, n_summary and the dataset size.

In [ ]:
def run(fixed, priors, data, n_summary=None, epochs=200, batch_size=64, lr=1e-3):
    n_params = data["P_train"].shape[1]
    n_summary = n_summary or n_params + 2

    print("\nTraining...\n")
    model, loss_history = train(data["X_train"], data["N_train"], data["P_train"],
                  n_summary=n_summary, epochs=epochs,
                  batch_size=batch_size, lr=lr)

    plot_loss(loss_history)
    save_model(model, loss_history, fixed, priors, n_summary)

    # evaluating
    print("\n" + 100*"-" + "\nEvaluating on test set...")
    s, x_hat = evaluate(model, data["X_test"], data["N_test"],
                        data["P_test"], data["stats"], fixed, priors, Phase=data.get("Phase_test"))

    return model, s, x_hat

def experiment(fixed, priors, n_train, n_test, n_summary, epochs, train_model=True, show_plots=True):
    # generate dataset for training and test
    generate_train_test_datasets(fixed, priors, n_train=n_train, n_test=n_test)
    data = load_train_test_datasets(fixed, priors, n_train=n_train, n_test=n_test)

    if train_model:
        # train and evaluate model
        model, s, x_hat = run(fixed, priors, data, n_summary=n_summary, epochs=epochs)
    else:
        # load model
        model, loss_history = load_model(fixed, priors, n_summary=n_summary)
        if show_plots:
            plot_loss(loss_history)
        s, x_hat = evaluate(model, data["X_test"], data["N_test"],
                            data["P_test"], data["stats"], fixed, priors, Phase=data.get("Phase_test"),
                            show_plots=show_plots)
    return model, s, x_hat

## 1. Exploring the Parameter Space

Before any training we run the model a bit first to get a
feel for its behavior and scan the (tau, T) plane with the other parameters
fixed at plausible values (Nd=11, sigma=0.05, Bmax=5).

**What the grid shows.** All simulations oscillate with a dominant cycle, but the *amplitude
modulation* changes regime: for small delay T the envelope is regular (limit-cycle /
multiperiodic), while increasing T (especially relative to tau) makes the modulation
increasingly irregular → chaos. This is expected: the delay is the source of chaos in this
model, and with sigma>0 noise and chaos can be hard to tell.

**Why this matters.** This grid is how we selected the "safe" (non-chaotic) operating point
tau=2.8, T=3.7 and the restricted priors used in the warm-up experiments below, following
the agreed stepwise strategy: first make ENCA work in a benign regime, then push toward
chaos.


In [ ]:
%matplotlib inline
list_tau, list_T = [1, 2, 3, 4, 5], [1, 3, 5, 7]
fig,axes = plt.subplots(len(list_tau),len(list_T),
                        figsize=(15,10))
for i,tau in enumerate(list_tau):
    for j,T in enumerate(list_T):
        params = [tau, T, 11, 0.05, 5]
        x, _ = simulate(params)
        axes[i,j].plot(x, label=f"tau={tau}, T={T}", lw=2)
        axes[i,j].set_title(f"tau={tau}, T={T}")

plt.tight_layout()
plt.show()

---
## 2. One varying Parameter - $N_d$

**Setup.** Warm-up experiment in the simplest configuration:
only the dynamo number varies, $Nd$ ~ $U(10, 12)$, everything else fixed at the safe operating
point; time domain, biLSTM decoder, n_summary = 2 (1 parameter + 1 free dimension),
150k training samples.

**Results.**
- Loss curves: both reconstruction and regression converge smoothly.
- Reconstruction: essentially on top of the original — with the noise given, the decoder
  has effectively learned the model dynamics in this regime.
- Regression check: s_0 vs Nd lies on the y=x line — a single learned summary statistic
  recovers the dynamo number almost perfectly.
- **Latent space: the most interesting finding.** The test points split into TWO separate
  clusters/branches, each with a smooth Nd gradient. The encoder is not told anything about
  regimes, yet it spends its free dimension separating two families of realisations.
  Hypothesis: in this intermediate Nd range the model is
  *bistable* — for the SAME parameters a realisation can end up in different dynamical
  behaviors (e.g. modulated/weak vs regular/strong cycle), and the parameter estimator alone
  cannot tell you which one occurred. The free summary captures exactly this. Sections 2a-2b
  test this hypothesis.



In [ ]:
%matplotlib widget
USE_FOURIER = False # if False, the Time Space is used for X
USE_FOURIER_NOISE = False # if True, the noise will be converted to Fourier space
priors = {
    "tau": (1,5),
    "T": (1,3),
    "Nd": (10,12),
    "sigma": (0,0.3),
    "Bmax": (5,5)
}
fixed = {
    "tau": 2.8,
    "T": 3.7,
    "Nd": None,
    "sigma": 0.05,
    "Bmax": 5
}

model, s, x_hat = experiment(fixed, priors,
                              n_train=150000, n_test=5000,
                              n_summary=2, epochs=200,
                              train_model=False)

### 2a. Clusters Inspection
Now let's see how the two clusters differ in the simulations.

**Method.** We split the latent space with a simple linear separator (found by eye), then
pick representative test samples from each cluster at evenly spaced quantiles of s_0 and
plot the corresponding raw simulations side by side; a companion scatter shows where the
chosen samples sit in latent space.

**What to look for / result.** The two rows should display systematically different
dynamics for overlapping values of Nd — e.g. one cluster with strong regular cycles and the
other with weaker, irregularly modulated cycles. That is direct visual evidence that the
free summary s_1 encodes the *realised dynamical regime* rather than a parameter: exactly
the bistability conjectured before. Note that the separator is hand-tuned for the
specific trained model loaded here; if the model is retrained the line (and even the number
of clusters) can change, since nothing in the loss enforces this structure.


In [ ]:
def get_cluster_points(s_cluster, full_indices, n_points=5, quantiles=None):
    """
    Pick n_points from a cluster at evenly-spaced quantiles of s0.
    Returns the indices into the *original* s array.
    """
    if quantiles is None:
        quantiles = np.linspace(0.10, 0.90, n_points)
    thresholds = np.quantile(s_cluster[:, 0], quantiles)
    chosen = []
    for q in thresholds:
        # pick the point whose s0 is closest to the target quantile
        idx_local = np.argmin(np.abs(s_cluster[:, 0] - q))
        chosen.append(full_indices[idx_local])
    return np.array(chosen)


def fetch_simulation(idx, data):
    """Return the un-normalised time-series for sample idx from the test set."""
    stats  = data["stats"]
    x_norm = data["X_test"][idx]
    return x_norm * stats["x_std"] + stats["x_mean"]


data = load_train_test_datasets(fixed, priors, n_train=150000, n_test=5000)
model, loss_history = load_model(fixed, priors, n_summary=2)
Params_real = data["P_test"]*data["stats"]["p_std"] + data["stats"]["p_mean"]

# get separator between the two clusters
x = np.linspace(-1.5,1.2,100)
y = -1.1*x-0.2
mask_cluster = -(s[:,1]+0.2)*10/11>s[:,0]


N_POINTS = 7        # columns in the grid
QUANTILES = np.linspace(0.01, 0.99, N_POINTS)

# indices into the full test array for each cluster
idx_blue = np.where(mask_cluster)[0]
idx_red  = np.where(~mask_cluster)[0]

s_blue = s[mask_cluster]
s_red  = s[~mask_cluster]

chosen_blue = get_cluster_points(s_blue, idx_blue, N_POINTS, QUANTILES)
chosen_red  = get_cluster_points(s_red,  idx_red,  N_POINTS, QUANTILES)

# --- grid plot ------------------------------------------------------

fig, axes = plt.subplots(2, N_POINTS, figsize=(4 * N_POINTS, 6),
                         sharey=False, sharex=True)

cluster_info = [
    (chosen_blue, s_blue, idx_blue, "Blues_r", "Cluster A (blue)"),
    (chosen_red,  s_red,  idx_red,  "Reds",    "Cluster B (red)"),
]

for row, (chosen, s_cl, full_idx, cmap, label) in enumerate(cluster_info):
    cmap_fn = plt.get_cmap(cmap)
    # color each column by the position of that point within its cluster's s0 range
    s0_min, s0_max = s_cl[:, 0].min(), s_cl[:, 0].max()

    for col, sample_idx in enumerate(chosen):
        ax = axes[row, col]
        x_sim = fetch_simulation(sample_idx, data)
        s0_val = s[sample_idx, 0]
        s1_val = s[sample_idx, 1]
        nd_val  = Params_real[sample_idx, 0]

        # color by fractional position in s0 range (consistent with scatter)
        frac  = (s0_val - s0_min) / max(s0_max - s0_min, 1e-9)
        color = cmap_fn(0.2 + 0.6 * frac)
        color = "steelblue" if row==0 else "tomato"

        ax.plot(x_sim, lw=2, color=color)
        ax.set_title(
            f"$N_d$={nd_val:.2f}\n$s_0$={s0_val:.2f}  $s_1$={s1_val:.2f}",
            fontsize=8,
        )
        ax.tick_params(labelsize=7)
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0,350)
        ax.set_xlim(0,200)

        if col == 0:
            ax.set_ylabel(label + "\n$x$", fontsize=9)
        if row == 1:
            ax.set_xlabel("time step", fontsize=8)

plt.suptitle(
    "Representative simulations across s₀ range for each cluster\n"
    "(columns = evenly-spaced quantiles of $s_0$, low → high)",
    fontsize=11,
)
plt.tight_layout()
_show_static(fig)


# --- companion scatter: show which points were picked ---------------

fig2, ax2 = plt.subplots(figsize=(6, 4))

ax2.scatter(s_blue[:, 0], s_blue[:, 1], c="steelblue",  alpha=0.3, s=10, label="Cluster A")
ax2.scatter(s_red[:,  0], s_red[:,  1], c="tomato",     alpha=0.3, s=10, label="Cluster B")

# highlight the chosen points
ax2.scatter(s[chosen_blue, 0], s[chosen_blue, 1],
            c="navy", s=80, zorder=5, marker="*", label="Chosen A")
ax2.scatter(s[chosen_red,  0], s[chosen_red,  1],
            c="firebrick", s=80, zorder=5, marker="*", label="Chosen R")

# label each chosen point with its column index
for col, idx in enumerate(chosen_blue):
    ax2.annotate(str(col + 1), (s[idx, 0], s[idx, 1]),
                 textcoords="offset points", xytext=(4, 4), fontsize=7, color="navy")
for col, idx in enumerate(chosen_red):
    ax2.annotate(str(col + 1), (s[idx, 0], s[idx, 1]),
                 textcoords="offset points", xytext=(4, 4), fontsize=7, color="firebrick")

ax2.set_xlabel("$s_0$")
ax2.set_ylabel("$s_1$")
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)
ax2.set_aspect("equal")
plt.title("Latent space - chosen samples highlighted")
plt.tight_layout()
_show_static(fig2)

### 2b. Bistability Check in the Transition Region

**Direct test of the bistability hypothesis, independent of the autoencoder.** For each $Nd$
in [10, 12.5] we run several stochastic realisations with identical parameters (different
random seeds). If the system is bistable in this range, the same $Nd$ must produce visibly
different behaviors across seeds — confirming that the latent clusters of section 2
correspond to a property of the *model*, not an artifact of the network.

**Result.** In the transition region different seeds with the same $Nd$ do end up in different
regimes (regular strong cycle vs modulated/weaker one), while at the edges of the range the
behavior is consistent across seeds. This closes the loop: parameter → bistable dynamics ->
extra summary dimension needed to describe a realisation.



In [ ]:
Nd_values = np.linspace(10.0, 12.5, 10)
n_runs_per_Nd = 4
tau_bs, T_bs, sigma_bs, Bmax_bs = 2.8, 3.7, 0.05, 5

fig, axes = plt.subplots(len(Nd_values), 1,
                          figsize=(10, 16), sharex=True)
for i, Nd in enumerate(Nd_values):
    for j in range(n_runs_per_Nd):
        x, _ = simulate([tau_bs, T_bs, Nd, sigma_bs, Bmax_bs])
        axes[i].plot(x, lw=1)
        axes[i].set_title(f"Nd={Nd:.2f}", fontsize=8)
        axes[i].tick_params(labelsize=6)
        axes[i].set_ylim(0,350)
        axes[i].set_xlim(0,200)
axes[-1].set_xlabel("time step")
plt.suptitle("Multiple stochastic realisations per Nd value (same parameters, different random seed)")
plt.tight_layout()
plt.show()

### 2c. Wider prior $N_d \in [2.5, 15]$ (time domain)

Stress test of the single-parameter case: the prior now spans weak/decaying, bistable and
strong/regular regimes at once (and fewer samples, 20k). Compared with the narrow-prior run,
the regression degrades in the regions where realisations are irregular: in the time domain
the pointwise MSE cannot reconstruct irregular envelopes, so the encoder gets a weaker
learning signal there. It is the first hint of the limitation that section 5 exposes
systematically.


In [ ]:
# using another prior range for Nd
USE_FOURIER = False # if False, the Time Space is used for X
priors = {
    "tau": (1,5),
    "T": (1,3),
    "Nd": (2.5,15),
    "sigma": (0,0.3),
    "Bmax": (5,5)
}
fixed = {
    "tau": 2.8,
    "T": 3.7,
    "Nd": None,
    "sigma": 0.05,
    "Bmax": 5
}

model, s, x_hat = experiment(fixed, priors,
                              n_train=20000, n_test=5000,
                              n_summary=2, epochs=200,
                              train_model=False)

### 2d. Same single-parameter problem, but in Fourier space

First trial of the Fourier representation on the easy problem ($Nd$ in [10,12]), to verify the
pipeline (FFT → log-magnitude → CNN decoder → inverse FFT for visualization) before using
it where it is actually needed. Performance is comparable to the time-domain run — as
expected, in the benign regime both representations work.

In [ ]:
# using Fourier transform
USE_FOURIER = True # if False, the Time Space is used for X
priors = {
    "tau": (1,5),
    "T": (1,3),
    "Nd": (10,12),
    "sigma": (0,0.3),
    "Bmax": (5,5)
}
fixed = {
    "tau": 2.8,
    "T": 3.7,
    "Nd": None,
    "sigma": 0.05,
    "Bmax": 5
}

model, s, x_hat = experiment(fixed, priors,
                              n_train=15000, n_test=5000,
                              n_summary=2, epochs=200,
                              train_model=False)

### 2e. Wide prior $N_d \in [0, 15]$ + Fourier

Fourier counterpart of 2c: full $Nd$ range including the dead-dynamo region ($Nd$ below the
critical dynamo number, where $B$ decays and the spectrum is essentially noise). The spectrum
separates the dominant cycle from irregular modulation, so the regression remains informative
over a wider range than in 2c; for very low $Nd$ there is little signal to regress on, and no
representation can fix that.


In [ ]:
# using another prior range for Nd + Fourier
USE_FOURIER = True # if False, the Time Space is used for X
priors = {
    "tau": (1,5),
    "T": (1,3),
    "Nd": (0,15),
    "sigma": (0,0.3),
    "Bmax": (5,5)
}
fixed = {
    "tau": 2.8,
    "T": 3.7,
    "Nd": None,
    "sigma": 0.05,
    "Bmax": 5
}

model, s, x_hat = experiment(fixed, priors,
                              n_train=20000, n_test=5000,
                              n_summary=2, epochs=200,
                              train_model=False)

---
## 3. Two Varying Parameters

Next step of the incremental strategy: two parameters at a time, still in the safe regime,
time domain, n_summary = 3 (2 parameters + 1 free).

### 3a. $N_d$ and $\sigma$

**Result.** Both parameters are recovered, with two caveats observed in the experiments:
- accuracy degrades for larger sigma — more stochastic forcing means each realisation is
  less informative about the deterministic part of the dynamics, so this is expected and is
  itself a meaningful observation (sigma controls the signal-to-noise of the problem);
- estimating sigma is intrinsically harder than $Nd$ from a single finite series: it is a
  variance-like quantity, well determined only "on average".
In the latent space the two regressed dimensions again organise on (two) sheet-like
manifolds, with the free dimension separating them — same bistability signature as section 2.

In [ ]:
USE_FOURIER = False
USE_FOURIER_NOISE = False
priors = {
    "tau": (1,5),
    "T": (1,3),
    "Nd": (10,12),
    "sigma": (0,0.3),
    "Bmax": (5,5)
}
fixed = {
    "tau": 2.8,
    "T": 3.7,
    "Nd": None,
    "sigma": None,
    "Bmax": 5
}

model, s, x_hat = experiment(fixed, priors,
                              n_train=20000, n_test=2000,
                              n_summary=3, epochs=200,
                              train_model=False)

### 3b. $\tau$ and $T$

The two *temporal* parameters vary (the physically interesting combination: one of them alone just rescales the time axis, while their ratio is the
dimensionless quantity that drives the dynamics, together with $Nd$).

**Result.** Good recovery of both, with one instructive artifact: the $s_1$ vs $T$ scatter shows
a *staircase* pattern. This is NOT a network failure — the simulator rounds $T$ to a multiple
of the integration step ($T = round(T/DT)*DT$, $DT=0.1$) so that the delay is an integer number
of grid steps. $T$ is effectively a discrete variable on a 0.1 grid, and the encoder honestly
learns those plateaus.


In [ ]:
USE_FOURIER = False
USE_FOURIER_NOISE = False
priors = {
    "tau": (1,5),
    "T": (1,3),
    "Nd": (10,12),
    "sigma": (0,0.3),
    "Bmax": (5,5)
}
fixed = {
    "tau": None,
    "T": None,
    "Nd": 11,
    "sigma": 0.05,
    "Bmax": 5
}

model, s, x_hat = experiment(fixed, priors,
                              n_train=20000, n_test=2000,
                              n_summary=3, epochs=200,
                              train_model=False)

---
## 4. All Four Varying Parameters - $\tau$, $T$, $N_d$, $\sigma$

Full problem (Bmax stays fixed — it only rescales the amplitude),
still with restricted priors away from chaos; time domain, n_summary = 6 (4 + 2 free).

**Result.** The temporal parameters $tau$ and $T$ are learned well; $Nd$ and sigma are noticeably
harder.
Plausible reading: with four parameters varying jointly, different ($Nd$, $sigma$) combinations
can produce nearly indistinguishable realisations within the restricted prior — a partial
identifiability problem — while $tau$ and $T$ leave a clear fingerprint (period of the dominant
cycle).


In [ ]:
USE_FOURIER = False
USE_FOURIER_NOISE = False
priors = {
    "tau": (1,5),
    "T": (1,3),
    "Nd": (10,12),
    "sigma": (0,0.3),
    "Bmax": (5,5)
}
fixed = {
    "tau": None,
    "T": None,
    "Nd": None,
    "sigma": None,
    "Bmax": 5
}

model, s, x_hat = experiment(fixed, priors,
                              n_train=20000, n_test=2000,
                              n_summary=6, epochs=200,
                              train_model=False)

---
## 5. Pushing Toward Chaos

Controlled breakdown experiment: same setting as 3b ($tau$ and $T$ vary) but progressively
extending the prior of the delay $T$ into the chaotic region. The point is to *show*, not just
claim, why the time-domain approach fails: the loss compares trajectories point by point,
and a chaotic realisation cannot be matched point by point — tiny phase errors produce huge
MSE, and reproducing a specific chaotic
realisation is essentially impossible.

### 5a. $T\in [1,4]$

Mild extension: a fraction of the realisations is now chaotic. Reconstructions of the
regular samples are still fine, the irregular ones degrade, and the regression starts to
blur correspondingly.


In [ ]:
USE_FOURIER = False
USE_FOURIER_NOISE = False
priors = {
    "tau": (1,5),
    "T": (1,4),
    "Nd": (10,12),
    "sigma": (0,0.3),
    "Bmax": (5,5)
}
fixed = {
    "tau": None,
    "T": None,
    "Nd": 11,
    "sigma": 0.05,
    "Bmax": 5
}

model, s, x_hat = experiment(fixed, priors,
                              n_train=20000, n_test=2000,
                              n_summary=3, epochs=200,
                              train_model=False)

### 5b. $T\in[1,7]$

Strong extension into chaos. Reconstructions of chaotic realisations clearly fail in the
time domain, and the parameter regression deteriorates: the network gets "confused by being
trained on chaos" because a large part of the training signal is pointwise
unpredictable.


In [ ]:
USE_FOURIER = False
USE_FOURIER_NOISE = False
priors = {
    "tau": (1,5),
    "T": (1,7),
    "Nd": (10,12),
    "sigma": (0,0.3),
    "Bmax": (5,5)
}
fixed = {
    "tau": None,
    "T": None,
    "Nd": 11,
    "sigma": 0.05,
    "Bmax": 5
}

model, s, x_hat = experiment(fixed, priors,
                              n_train=20000, n_test=5000,
                              n_summary=3, epochs=200,
                              train_model=False)

The method works well in the non-chaotic regime, but if we push it towards chaos, it breaks down.\
For this reason, the Fourier space can be a potential solution.

# 6. Fourier Space
Apply FFT to $X$ before the encoder.

The autoencoder now works entirely on log-magnitude spectra: the encoder summarises the
spectrum, the decoder (CNN — the LSTM makes no sense without a time axis) reconstructs the
spectrum from (s, noise). The training loss is computed in Fourier space; for visualization
we also back-project reconstructions to the time domain using the stored phase of the
original signal. Note: losses in Fourier and time space are NOT directly
comparable numbers — comparisons must be made on reconstructions in the same space.

### 6a. $\tau$ and $T$, with $T\in[1,7]$

Direct re-run of the failure case 5b in Fourier space.

**Result.** Clear improvement: the spectra are well reconstructed (dominant peak captured —
the quantity that matters), the back-projected time series match
the dominant cycle even for irregular realisations, and the $\frac{tau}{T}$ regression stays clean over
the whole extended prior — including chaos. Exactly the predicted mechanism: the spectrum
separates "signal" (peak: position/height ← parameters) from "chaos" (broadband background),
so the loss no longer punishes the network for the unpredictable part.



In [ ]:
USE_FOURIER = True
USE_FOURIER_NOISE = False
USE_FNO = False
priors = {
    "tau": (1,5),
    "T": (1,7),
    "Nd": (10,12),
    "sigma": (0,0.3),
    "Bmax": (5,5)
}
fixed = {
    "tau": None,
    "T": None,
    "Nd": 11,
    "sigma": 0.05,
    "Bmax": 5
}

model, s, x_hat = experiment(fixed, priors,
                              n_train=20000, n_test=2000,
                              n_summary=3, epochs=200,
                              train_model=False)

### 6b. All four parameters, restricted prior

Fourier-space version of section 4 (same restricted priors), to isolate the effect of the
representation with all four parameters varying. Performance is comparable or slightly
better than the time-domain run; the real advantage of Fourier shows up when the prior
includes chaos (6c).


In [ ]:
USE_FOURIER = True
USE_FOURIER_NOISE = False
priors = {
    "tau": (1,5),
    "T": (1,3),
    "Nd": (10,12),
    "sigma": (0,0.3),
    "Bmax": (5,5)
}
fixed = {
    "tau": None,
    "T": None,
    "Nd": None,
    "sigma": None,
    "Bmax": 5
}

model, s, x_hat = experiment(fixed, priors,
                              n_train=20000, n_test=2000,
                              n_summary=6, epochs=200,
                              train_model=False)

### 6c. All four parameters, full prior range (paper priors)

**The main result of the project.** Four varying parameters with the FULL priors of the
inference paper (150k training samples, Fourier representation, n_summary = 6):
 - $tau$ 
 - $T$ in (0.1, 10)
 - $Nd$ in (0, 15)
 - $sigma$ in (0, 0.3) 


**Result.** Good spectrum reconstructions and a working regression over the full prior. Residual structure in the
extra summaries: projections of the 6-D latent space (e.g. s4/s5 vs the parameter-mapped
dimensions) still show two manifolds — the regime/bistability information again, now in the
full-complexity setting.


In [ ]:
USE_FOURIER = True
USE_FOURIER_NOISE = False
USE_FNO = False
priors = {
    "tau": (0.1,10),
    "T": (0.1,10),
    "Nd": (0,15),
    "sigma": (0,0.3),
    "Bmax": (5,5)
}
fixed = {
    "tau": None,
    "T": None,
    "Nd": None,
    "sigma": None,
    "Bmax": 5
}

model, s, x_hat = experiment(fixed, priors,
                              n_train=150000, n_test=5000,
                              n_summary=6, epochs=200,
                              train_model=False)

### 6d. All four parameters, full prior range (paper priors), n_summary = n_params = 4

**Ablation: are the extra summary
dimensions actually needed?** Same setting as 6c but with the bottleneck squeezed to
n_summary = n_params = 4 (and a smaller dataset, 20k, for computational reasons — keep this
in mind when comparing with 6c).

**Expected/observed.** With no free dimensions the encoder must either sacrifice regime
information (worse reconstruction, since the decoder can't know which behavior to rebuild
in the bistable range) or contaminate the parameter estimators with it (worse regression).
The latent clusters of 2/6c are the smoking gun that the model needs at least one dimension
beyond the parameters. This connects back to theory: parameter estimators alone are not
sufficient statistics for a single finite realisation.


In [ ]:
USE_FOURIER = True
USE_FOURIER_NOISE = False
USE_FNO = False
priors = {
    "tau": (0.1,10),
    "T": (0.1,10),
    "Nd": (0,15),
    "sigma": (0,0.3),
    "Bmax": (5,5)
}
fixed = {
    "tau": None,
    "T": None,
    "Nd": None,
    "sigma": None,
    "Bmax": 5
}

model, s, x_hat = experiment(fixed, priors,
                              n_train=20000, n_test=2000,
                              n_summary=4, epochs=200,
                              train_model=False)

# 7. Fourier noise

Here we try represent the *noise* in Fourier space
too, so that decoder input and output live in the same space. FFT is invertible, so a specific noise realisation carries the same information
in either representation, and the dynamo equations actually look simpler in frequency space
(time derivatives become multiplications), so the mapping: spectrum(noise) → spectrum(x)
might be easier to learn. Our initial doubt ("the spectrum of white noise is flat") applies
to the *expected* spectrum, not to a single realisation, which keeps its fluctuations.
Caveat: we transform only the magnitude — the noise phase is discarded, so unlike the
invertible-FFT argument, the decoder here actually receives *less* information than in
time domain.

**Result.** [Compare with 6c, which differs only in the noise representation: same priors,
150k samples, n_summary=6.] In our runs it did not bring a clear improvement over time-domain
noise. Plausible explanation: dropping the noise phase removes the timing information that
the decoder would need to place the modulation, so the magnitude-only noise spectrum is a
weaker conditioning signal.


In [ ]:
USE_FOURIER = True
USE_FOURIER_NOISE = True
USE_FNO = False
priors = {
    "tau": (0.1,10),
    "T": (0.1,10),
    "Nd": (0,15),
    "sigma": (0,0.3),
    "Bmax": (5,5)
}
fixed = {
    "tau": None,
    "T": None,
    "Nd": None,
    "sigma": None,
    "Bmax": 5
}

model, s, x_hat = experiment(fixed, priors,
                              n_train=150000, n_test=5000,
                              n_summary=6, epochs=200,
                              train_model=False)

## 8. Using FNO

The last experiment is about replacing the decoder with a
**Fourier Neural Operator**. FNOs are built to learn *solution operators* of
(stochastic) differential equations — mappings from an input function (here: the noise
realisation, conditioned on s) to an output function (the trajectory/spectrum). Each FNO
block mixes a limited number of low-frequency modes globally (learned complex matrices) and
adds a local pointwise bypass; this global receptive field matches the delayed, oscillatory
structure of the dynamo far better than purely local convolutions.

Same full-prior setting as 6c (150k samples, n_summary=6, fewer epochs: 100).

**Result.** [Judge against 6c: spectrum reconstructions, regression scatter, latent
structure.] If quality is on par with (or better than) the CNN decoder with half the
training epochs, that is the message: the operator-learning inductive bias fits this
problem.

In [ ]:
USE_FOURIER = True
USE_FOURIER_NOISE = False
USE_FNO = True
priors = {
    "tau": (0.1,10),
    "T": (0.1,10),
    "Nd": (0,15),
    "sigma": (0,0.3),
    "Bmax": (5,5)
}
fixed = {
    "tau": None,
    "T": None,
    "Nd": None,
    "sigma": None,
    "Bmax": 5
}

model, s, x_hat = experiment(fixed, priors,
                              n_train=150000, n_test=5000,
                              n_summary=6, epochs=100,
                              train_model=False)

---
# Conclusions

1. **ENCA works on the solar dynamo.** A CNN encoder + noise-conditioned decoder learns
   interpretable, near-minimal summary statistics for the SDDE dynamo model: in the benign
   regime the parameter-mapped summaries sit on the y=x line.
2. **The representation is decisive.** In the time domain the method breaks down as the
   delay pushes the model into chaos (pointwise loss vs unpredictable trajectories). In
   Fourier space — where chaos becomes broadband background and the parameter information
   is concentrated in the dominant peak — the full priors of the inference paper become
   tractable (with ~150k simulations), a setting where previous time-domain attempts failed.
3. **The free summary dimensions are physical.** They encode the realised dynamical regime
   (bistability: same parameters, different behaviors), which parameter estimators alone
   cannot capture — verified both via latent-space clusters and direct multi-seed
   simulations. Consistently, squeezing the bottleneck to n_summary = n_params degrades
   performance.
4. **Outlook.** FNO decoders (operator learning) as a natural next architecture; applying
   the trained encoder to the real sunspot record to check whether the data falls inside
   the prior range of the model (the agreed stopping point of the project — full Bayesian
   inference with these summaries is the natural continuation).

## RUN THIS CELL ONLY IN GOOGLE COLAB TO SAVE MODELS

In [ ]:
import shutil
from google.colab import files

for i,name in enumerate(["data", "models"]):
  shutil.make_archive(f'downloaded_files_{i+1}', 'zip', name)
  files.download(f'downloaded_files_{i+1}.zip')